# Liver Disease Prediction — Exploratory Data Analysis

**Dataset**: [Predict Liver Disease — 1700 Records](https://www.kaggle.com/datasets/rabieelkharoua/predict-liver-disease-1700-records-dataset)  
**Features**: 10 clinical + demographic features, binary target (`Diagnosis`)  
**Goal**: Understand distributions, relationships, and predictive signal before modelling.

All logic lives in `src/` — this notebook is pure narrative + visualization.

In [ ]:
import sys
from pathlib import Path

# Allow imports from project root
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from src.config import load_settings
from src.data.loader import load_raw_data
from src.data.validation import run_all_validations
from src.features.schema import CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET
from src.features.woe import iv_summary_table
from src.features.woe_viz import plot_iv_comparison, plot_woe_bars, validate_woe_monotonicity
from src.visualization.eda_plots import (
    crosstabulate,
    plot_boxplot,
    plot_correlation_matrix,
    plot_percentage_stacked_chart,
)

settings = load_settings()
print('Setup complete.')

## 1. Load & Validate Data

In [ ]:
df = load_raw_data(settings)
print(f'Shape: {df.shape}')
print(f'\nTarget distribution:')
print(df[TARGET].value_counts())
print(f'\nPositive rate: {(df[TARGET] == "Positive").mean():.2%}')

In [ ]:
warnings = run_all_validations(df)
if warnings:
    print('Validation warnings:')
    for w in warnings:
        print(f'  - {w}')
else:
    print('All validations passed. No issues found.')

In [ ]:
df.head()

## 2. Descriptive Statistics

In [ ]:
df[NUMERIC_FEATURES].describe().round(2)

In [ ]:
# Group by diagnosis
df.groupby(TARGET)[NUMERIC_FEATURES].mean().round(2)

## 3. Numeric Features vs Diagnosis

Boxplots show the distribution of each numeric feature split by target class.  
Welch's t-test p-values are annotated: `***` p<0.001, `**` p<0.01, `*` p<0.05, `ns` not significant.

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(16, 4))
for ax, feat in zip(axes, NUMERIC_FEATURES):
    plot_boxplot(df, feat, TARGET, ax=ax)
fig.suptitle('Numeric Features vs Diagnosis', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

## 4. Categorical Features vs Diagnosis

100% stacked bar charts show the proportion of Positive/Negative diagnoses within each category.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()
for i, feat in enumerate(CATEGORICAL_FEATURES):
    ct = crosstabulate(df, feat, TARGET, bins=6)
    plot_percentage_stacked_chart(ct, title=feat, xlabel=feat, ax=axes_flat[i])
for j in range(len(CATEGORICAL_FEATURES), len(axes_flat)):
    axes_flat[j].set_visible(False)
fig.suptitle('Categorical Features vs Diagnosis', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 5. Statistical Tests

Welch's t-tests for numeric features, chi-square tests for categorical features.

In [ ]:
results = []
pos = df[df[TARGET] == 'Positive']
neg = df[df[TARGET] == 'Negative']

for feat in NUMERIC_FEATURES:
    t, p = stats.ttest_ind(pos[feat].dropna(), neg[feat].dropna(), equal_var=False)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    results.append({'Feature': feat, 'Test': "Welch's t-test", 't/chi2': round(t, 3), 'p-value': round(p, 4), 'Sig': sig})

for feat in CATEGORICAL_FEATURES:
    ct = pd.crosstab(df[feat], df[TARGET])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    results.append({'Feature': feat, 'Test': 'Chi-square', 't/chi2': round(chi2, 3), 'p-value': round(p, 4), 'Sig': sig})

pd.DataFrame(results).set_index('Feature')

## 6. Correlation Matrix (Numeric Features)

Pearson correlations between numeric features. Low multicollinearity is ideal for linear models.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_correlation_matrix(df, NUMERIC_FEATURES, ax=ax)
fig.tight_layout()
plt.show()

## 7. Weight of Evidence (WoE) Analysis

WoE measures the log-odds of a feature bin being associated with the positive class.  
Information Value (IV) quantifies the total predictive power of each feature.

| IV Range | Predictive Power |
|----------|------------------|
| < 0.02   | Useless          |
| 0.02–0.1 | Weak             |
| 0.1–0.3  | Medium           |
| 0.3–0.5  | Strong           |
| > 0.5    | Suspicious       |

In [ ]:
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
iv_df = iv_summary_table(df, all_features, TARGET)
iv_df.sort_values('IV', ascending=False).reset_index(drop=True)

In [ ]:
# IV comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
plot_iv_comparison(df, all_features, TARGET, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# WoE bar charts for top 3 features by IV
top3 = iv_df.sort_values('IV', ascending=False).head(3)['Feature'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, top3):
    plot_woe_bars(df, feat, TARGET, ax=ax)
fig.suptitle('WoE by Bin — Top 3 Features', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## 8. WoE Monotonicity Validation

For ordinal features (GeneticRisk) we expect a monotonic WoE trend.  
Non-monotonic patterns may indicate binning issues or noise.

In [ ]:
result = validate_woe_monotonicity(
    df, 'GeneticRisk', TARGET,
    categories=['Low', 'Medium', 'High']
)
print(f"GeneticRisk WoE monotonicity:")
print(f"  Is monotonic : {result['is_monotonic']}")
print(f"  Direction    : {result['direction']}")
print(f"  WoE values   : {result['woe_values']}")
if result['violations']:
    print(f"  Violations   : {result['violations']}")

## 9. Key Findings Summary

In [ ]:
pos_rate = (df[TARGET] == 'Positive').mean()
top_feat = iv_df.sort_values('IV', ascending=False).iloc[0]

print('=== Key Findings ===')
print(f'Dataset: {len(df):,} records, {pos_rate:.1%} positive cases')
print(f'Top predictor: {top_feat["Feature"]} (IV = {top_feat["IV"]:.3f} — {top_feat["Predictive Power"]})')
print(f'All numeric features are statistically significant (t-test p < 0.001)')
print(f'No missing values or duplicates')
print(f'Low multicollinearity — suitable for linear models')